In [1]:
# 导入与环境设置
import os
import sys
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# 将项目根目录加入 sys.path 以便导入
PROJECT_ROOT = "/mnt/project_rlinf/jlchen/code/UniVLA"
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# 常量：默认数据与检查点路径（可按需修改或通过环境变量传入）
DATA_ROOT = os.environ.get("UNI_VLA_DATA_ROOT", "/mnt/project_rlinf/jlchen/datasets")
MIX_NAME  = "bridge_dataset"
# MIX_NAME  = "lam_plus"
CKPT_PATH = "/mnt/project_rlinf/jlchen/code/UniVLA/latent_action_model/logs/jepa_bridge_v2_16/version_0/checkpoints/epoch=15.ckpt"

# RLDS/Imagenet 标准化参数（与数据加载保持一致）
_IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)
_IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)

def unnormalize_imagenet(img_chw: torch.Tensor) -> np.ndarray:
    """
    输入: [C, H, W]，输出: [H, W, C] 的 uint8 numpy 图像
    """
    assert img_chw.ndim == 3 and img_chw.size(0) == 3
    img = img_chw.detach().cpu().float().numpy()
    img = (img * _IMAGENET_STD + _IMAGENET_MEAN).clip(0.0, 1.0)
    img = (img.transpose(1, 2, 0) * 255.0).round().astype(np.uint8)
    return img

def to_pil(img_chw: torch.Tensor) -> Image.Image:
    return Image.fromarray(unnormalize_imagenet(img_chw))

print(f"DATA_ROOT={DATA_ROOT}")
print(f"MIX_NAME={MIX_NAME}")
print("请设置 CKPT_PATH 环境变量 UNI_VLA_LAM_CKPT 指向 LAM 检查点文件" if not CKPT_PATH else f"CKPT_PATH={CKPT_PATH}")


DATA_ROOT=/mnt/project_rlinf/jlchen/datasets
MIX_NAME=bridge_dataset
CKPT_PATH=/mnt/project_rlinf/jlchen/code/UniVLA/latent_action_model/logs/jepa_bridge_v2_16/version_0/checkpoints/epoch=15.ckpt


In [2]:
# 加载 LAM 模型检查点
from latent_action_model.core.lam_model import load_latent_action_model

assert CKPT_PATH and os.path.exists(CKPT_PATH), (
    "未找到检查点，请设置环境变量 UNI_VLA_LAM_CKPT 或直接修改 CKPT_PATH 变量"
)

lam = load_latent_action_model(CKPT_PATH,vision_model_id="jepa")
lam.eval().to("cuda").to(torch.bfloat16)
model_device = next(lam.parameters()).device
print(f"LAM 已加载到: {model_device}")


TypeError: load_latent_action_model() got an unexpected keyword argument 'vision_model_id'

In [ ]:
# 构建并取样 LAM 训练数据（Open-X Embodiment RLDS）
from latent_action_model.genie.dataset import LightningOpenX

# 构建 DataModule 并进入 fit 阶段，使用与训练一致的 batch_transform/collator
openx_dm = LightningOpenX(
    data_root=DATA_ROOT,
    data_mix=MIX_NAME,
    batch_size=8,
    resolution=256,
    num_frames=16,
    episodic=False,
    shuffle_buffer_size=1024,
    image_aug=False,
)
openx_dm.setup(stage="fit")
train_loader = openx_dm.train_dataloader()




2025-11-03 18:11:22.061366: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2348] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 9.0. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
2025-11-03 18:11:22.063866: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2348] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 9.0. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
Load dataset info from /mnt/project_rlinf/jlchen/datasets/bridge_dataset/1.0.0
Constructing tf.data.Dataset bridge_dataset for split all, from /mnt/project_rlinf/jlchen/datasets/bridge_dataset/1.0.0
2025-11-03 18:11:36.276982: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization
[*] Loading existing dataset statistics from /mnt/project_rlinf/jlchen/datasets/bridge_dataset/1.0.0/dataset_statistics_edde9adadd0a1ea9ab8dca101d


######################################################################################
# Loading the following 1 datasets (incl. sampling weight):                         #
# bridge_dataset: ==========================================================1.000000 #
######################################################################################



2025-11-03 18:11:37.650847: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization
[*] Applying frame transforms on dataset...
Load dataset info from /mnt/project_rlinf/jlchen/datasets/bridge_dataset/1.0.0
Constructing tf.data.Dataset bridge_dataset for split all, from /mnt/project_rlinf/jlchen/datasets/bridge_dataset/1.0.0
2025-11-03 18:11:39.809371: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization
[*] Loading existing dataset statistics from /mnt/project_rlinf/jlchen/datasets/bridge_dataset/1.0.0/dataset_statistics_edde9adadd0a1ea9ab8dca101d68e2d8b2c50c27cbe12fa42c81472d15dbde80.json.
Constructing tf.data.Dataset bridge_dataset for split val, from /mnt/project_rlinf/jlchen/datasets/bridge_dataset/1.0.0
2025-11-03 18:11:40.087817: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization
[*] Threads per Dataset: [1]
[*] Re


######################################################################################
# Loading the following 1 datasets (incl. sampling weight):                         #
# bridge_dataset: ==========================================================1.000000 #
######################################################################################



2025-11-03 18:11:40.410287: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization
[*] Applying frame transforms on dataset...


In [ ]:
# 取一个 batch
batch = next(iter(train_loader))
print("batch keys:", list(batch.keys()))
print("videos:", batch["videos"].shape, "proprio:", batch["proprio"].shape)

batch keys: ['videos', 'task_instruction', 'action', 'proprio']
videos: torch.Size([8, 5, 3, 256, 256]) proprio: torch.Size([8, 5, 8])


In [ ]:
# 前向推理：得到 LAM 输出（含 delta_s_pred）
videos = batch["videos"].to(model_device).to(torch.bfloat16) # [B, T, C, H, W]
states = batch["proprio"].to(model_device).to(torch.bfloat16) # [B, T, state_dim]
with torch.no_grad():
    outputs = lam(videos,states)

# 解包输出（参见 LatentLAMModel._run 返回顺序）
recon, tgt, perplexity, indices, delta_s_pred, features, quantized, entropy_loss = outputs
print(
    f"delta_s_pred: {tuple(delta_s_pred.shape)} | perplexity: {perplexity} | indices: {indices.shape}"
)


delta_s_pred: (8, 8) | perplexity: 8.0 | indices: torch.Size([8, 4])


In [ ]:
# 计算监督信号（delta_s_gt）并与预测对比
# 约定：以 proprio 的前后两帧（位置前三维）差作为 delta_s_gt
proprio = batch["proprio"].float()  # [B, T, state_dim]
if proprio.ndim != 3 or proprio.size(-1) < 3:
    raise RuntimeError(f"无效的 proprio 形状: {tuple(proprio.shape)} (需要最后维度>=3)")

with torch.no_grad():
    delta_s_gt = proprio[:, -1] - proprio[:, 0]  # [B, 3]
    delta_s_pred_cpu = delta_s_pred.detach().cpu().float()

    # 指标：余弦相似度与 L2 误差
    cos_sim = F.cosine_similarity(delta_s_pred_cpu, delta_s_gt, dim=-1)
    l1_err = delta_s_pred_cpu - delta_s_gt
print(proprio[:,-1])
print(delta_s_gt[:,-1])
print(delta_s_pred_cpu[:,-1])
print("cos_sim:", cos_sim.tolist())
print("l1_err:", l1_err.tolist())

# 选择一个样本用于可视化
sample_idx = 0


tensor([[ 0.3498,  0.6024,  0.5264,  0.2265, -0.5332, -0.1402,  0.0000,  0.0495],
        [-0.2889, -0.4981,  0.6535, -0.2674, -0.0629,  0.0878,  0.0000, -0.8475],
        [ 0.2782, -0.2909,  0.4588, -0.6575, -0.2925,  0.0879,  0.0000,  0.9783],
        [-0.4514, -0.8501, -0.0865, -0.3668,  0.4050, -0.2607,  0.0000,  0.1388],
        [-0.5012,  0.1451,  0.3684,  0.3994,  0.0892, -0.2237,  0.0000,  0.9796],
        [ 0.0540, -0.7191,  0.3259,  0.1256,  0.2686, -0.0277,  0.0000,  0.9783],
        [-0.0695,  0.0975, -0.5728,  0.1879, -0.2201,  0.0455,  0.0000,  0.1034],
        [-0.5068, -0.3844,  0.2788, -0.2229,  0.1415, -0.1014,  0.0000,  0.0640]])
tensor([ 0.0000, -0.3749,  0.0000,  0.0748,  0.0000,  0.0000, -0.8679,  0.0000])
tensor([-0.0056,  0.0295,  3.8750, -0.0884,  4.3750,  4.0938, -3.9844, -0.0820])
cos_sim: [0.8737199306488037, 0.30520448088645935, 0.06470513343811035, -0.16321051120758057, 0.0986599400639534, 0.035459812730550766, 0.9928490519523621, 0.9573140144348145]
l1_er

In [ ]:
# 条件可视化：
# - 若 train_in_latent 且使用 Cosmos Autoencoder，则将视觉 token 解码为像素并显示
# - 若 train_in_latent 且非 Cosmos，则打印 recon 与 tgt 的特征余弦
# - 否则（非 latent），直接将 recon/tgt 还原为图像显示
if lam.train_in_latent:
    ve = getattr(lam, "vision_encoder", None)
    is_cosmos = (ve is not None) and hasattr(ve, "decode") and (ve.__class__.__name__.lower().startswith("cosmos"))
    if is_cosmos:
        with torch.no_grad():
            # 解码整个序列
            B, T, _, _ = features.shape   # [B, T, h*w, K]
            all_imgs11 = []
            for t in range(T):
                tokens_t = features[:, t]               # [B, h*w, K]
                imgs_t = ve.decode(tokens_t)            # [B, 3, H, W], [-1,1]
                all_imgs11.append(imgs_t.unsqueeze(1))  # [B,1,3,H,W]

            # 解码重建结果并追加到序列尾部
            recon_imgs11 = ve.decode(recon).unsqueeze(1)  # [B,1,3,H,W]
            all_imgs11.append(recon_imgs11)

            # 拼接成 [B, T+1, 3, H, W]
            all_imgs11 = torch.cat(all_imgs11, dim=1)

        def pil_from_neg1_to_1(img11: torch.Tensor):
            img01 = (img11.clamp(-1.0, 1.0) + 1.0) / 2.0
            img_uint8 = (img01 * 255.0).round().byte().cpu().permute(0, 2, 3, 1).numpy()
            return [Image.fromarray(arr) for arr in img_uint8]

        # ========== 批量可视化 ==========
        titles = [f'Frame {t}' for t in range(all_imgs11.shape[1] - 1)] + ['Reconstruction']
        num_rows, num_cols = all_imgs11.shape[0], all_imgs11.shape[1]

        plt.figure(figsize=(3 * num_cols, 3 * num_rows))

        for i in range(num_rows):
            imgs_pil = pil_from_neg1_to_1(all_imgs11[i])  # 转换该样本的所有帧
            for j, img in enumerate(imgs_pil):
                plt.subplot(num_rows, num_cols, i * num_cols + j + 1)
                plt.imshow(img)
                plt.axis('off')
                if i == 0:
                    plt.title(titles[j], fontsize=10)
        plt.tight_layout()
        plt.show()
    else:
        # 特征空间：将 [B, N, D] 或其他高维展平后做样本级余弦
        rec_flat = recon.reshape(recon.size(0), -1).detach().cpu().float()
        tgt_flat = tgt.reshape(tgt.size(0), -1).detach().cpu().float()
        cos_feat = F.cosine_similarity(rec_flat, tgt_flat, dim=-1)
        print("recon-tgt cosine (features):", cos_feat.tolist())
else:
    # 图像空间：逐行展示 batch 内样本；每行先显示所有 video 帧，最后一列仅显示一次 recon
    vids = videos.detach().cpu()
    recs = recon.detach().cpu()
    assert vids.ndim == 5, f"videos 期望为 [B,T,C,H,W]，实际为 {tuple(vids.shape)}"
    B, T, C, H, W = vids.shape

    # 统一处理 recon：支持 [B,T,C,H,W] 或 [B,C,H,W]
    if recs.ndim == 5:
        T_rec = recs.shape[1]
    elif recs.ndim == 4:
        T_rec = 1
    else:
        raise RuntimeError(f"recon 形状不支持: {tuple(recs.shape)} (支持 [B,T,C,H,W] 或 [B,C,H,W])")

    num_rows = B
    num_cols = T + 1  # T 列视频 + 1 列重建
    fig, axes = plt.subplots(num_rows, num_cols, figsize=(1.8 * num_cols, 1.8 * num_rows))
    if num_rows == 1:
        axes = axes.reshape(1, -1)

    for i in range(B):
        # 逐帧绘制视频
        for t in range(T):
            ax_vid = axes[i, t]
            ax_vid.imshow(to_pil(vids[i, t]))
            ax_vid.axis('off')
            if i == 0:
                ax_vid.set_title(f"t={t} Video", fontsize=9)

        # 最后一列绘制一次重建
        ax_rec = axes[i, T]
        if recs.ndim == 5:
            t_sel = min(T - 1, T_rec - 1)  # 若 recon 有时间维度，取最后可用一帧
            img_rec = recs[i, t_sel]
        else:
            img_rec = recs[i]
        ax_rec.imshow(to_pil(img_rec))
        ax_rec.axis('off')
        if i == 0:
            ax_rec.set_title("Recon", fontsize=9)

    plt.tight_layout()
    plt.show()


recon-tgt cosine (features): [0.7788172960281372, 0.8108011484146118, 0.7977308034896851, 0.7982417345046997, 0.7679286003112793, 0.7974792122840881, 0.820809006690979, 0.8041137456893921]
